# Hand Motion In-betweening — MANO 기반 손 동작 보간 (How2Sign)

SILK 구조 기반 baseline, Two-stage Transformer(Qin et al., SIGGRAPH Asia 2022), 그리고 그 확장 구조인
다중 키프레임(과제 3)까지 구현·평가한 노트북입니다.

## 1. 환경 설정

In [ ]:
!git clone https://github.com/JianHe0628/SignSparK.git
%cd /content/SignSparK
!pip install -r requirements.txt

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

## 2. 데이터 준비 (How2Sign, Google Drive 복원)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

LOCAL_DATA_ROOT = "/content/SignSparK/data"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/KUBIG/contest/HandMotionInbetweening_data_backup"

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
# CSL-Daily만 제외하고 나머지(How2Sign lmdb, index, mano)는 전부 복원
!rsync -avh --progress --exclude='CSL-Daily*' \
    "{DRIVE_BACKUP_DIR}/" "{LOCAL_DATA_ROOT}/"

## 3. 좌표계 변환 + LMDB 디코딩 유틸

In [ ]:
import numpy as np
import pickle, io, lmdb

# 왼손 좌표계 -> 오른손(WiLoR) 좌표계 변환 마스크 (R_flip=diag(1,-1,-1) 켤레변환과 동치)
_LEFT_HAND_FLIP_MASK = np.array([
    [ 1.0, -1.0, -1.0],
    [-1.0,  1.0,  1.0],
    [-1.0,  1.0,  1.0],
], dtype=np.float32)

def _rot6d_to_matrix_np(d6):
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = a1 / np.linalg.norm(a1, axis=-1, keepdims=True)
    b2 = a2 - np.sum(b1 * a2, axis=-1, keepdims=True) * b1
    b2 = b2 / np.linalg.norm(b2, axis=-1, keepdims=True)
    b3 = np.cross(b1, b2, axis=-1)
    return np.stack((b1, b2, b3), axis=-2)

def _matrix_to_rot6d_np(matrix):
    batch_dim = matrix.shape[:-2]
    return matrix[..., :2, :].copy().reshape(*batch_dim, 6)

def flip_left_hand_features(left_feats):
    left = left_feats.astype(np.float32, copy=False)
    T_len = left.shape[0]
    pose_6d = left[:, :90].reshape(T_len * 15, 6)
    mats = _rot6d_to_matrix_np(pose_6d) * _LEFT_HAND_FLIP_MASK
    flipped = _matrix_to_rot6d_np(mats).reshape(T_len, 90)
    if left.shape[-1] > 90:
        return np.concatenate([flipped, left[:, 90:]], axis=-1)
    return flipped

In [ ]:
# LMDB 값(클립별 npz 바이트) 디코딩
def deserialize_minimal_npz(data: bytes):
    with io.BytesIO(data) as buffer:
        npz_file = np.load(buffer, allow_pickle=True)
        return {
            "language": npz_file["language"][0],
            "translation": npz_file["translation"][0],
            "segment": npz_file["segment"],
            "left_features": npz_file["left_features"],
            "right_features": npz_file["right_features"],
            "body_features": npz_file["body_features"],
            "face_features": npz_file["face_features"],
        }

## 4. Index 파일 로드 (train / dev / test)

`train/dev/test_index.npz`가 각 윈도우의 (clip, hand, start[, T]) 조합을 미리 정의합니다.

In [ ]:
DRIVE_CONTEST_ROOT = "/content/drive/MyDrive/KUBIG/contest"

TRAIN_INDEX_PATH = f"{DRIVE_CONTEST_ROOT}/train_index.npz"
DEV_INDEX_PATH   = f"{DRIVE_CONTEST_ROOT}/dev_index.npz"
TEST_INDEX_PATH  = f"{DRIVE_CONTEST_ROOT}/test_index.npz"

for name, path in [("train", TRAIN_INDEX_PATH), ("dev", DEV_INDEX_PATH), ("test", TEST_INDEX_PATH)]:
    d = np.load(path, allow_pickle=True)
    print(f"[{name}] 윈도우 수: {len(d['clip_idx'])}, 클립 수: {len(d['clip_ids'])}, "
          f"필드: {d.files}")

## 5. Dataset / DataLoader

index 파일 기반으로 재현 가능하게 윈도우를 자르고, target 기준 상대위치(`rel_pos`)를 함께 반환합니다.

In [ ]:
import torch
import random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

_HAND_NAMES = {0: "left", 1: "right"}

_LMDB_ENV_CACHE = {}  # 같은 LMDB 경로는 env를 하나만 열어서 공유 (중복 open 에러 방지)

def _get_shared_lmdb_env(path):
    if path not in _LMDB_ENV_CACHE:
        _LMDB_ENV_CACHE[path] = lmdb.open(path, readonly=True, lock=False,
                                           readahead=False, meminit=False, max_readers=1024)
    return _LMDB_ENV_CACHE[path]


class IndexedHandInbetweeningDataset(Dataset):
    """train/dev/test_index.npz의 (clip_idx, hand, start[, T]) 조합으로 윈도우를 자르는 Dataset.

    mode="train": gap_len을 [5,30] 균등 랜덤 샘플링. mode="eval": T 필드를 gap_len으로 고정 사용.
    keyframe_strategy: None / "temporal_midpoint" / "segment2_priority" - 다중 키프레임 구조용.
    """

    def __init__(self, lmdb_path, index_path, mode="train",
                 context_len=10, target_len=1, gap_range=(5, 30),
                 keyframe_strategy=None, context_len_choices=None):
        assert mode in ("train", "eval")
        assert keyframe_strategy in (None, "temporal_midpoint", "segment2_priority")
        self.lmdb_path = lmdb_path
        self.mode = mode
        self.context_len = context_len
        self.target_len = target_len
        self.gap_range = gap_range
        self.keyframe_strategy = keyframe_strategy
        self.context_len_choices = context_len_choices
        self._env = None

        idx_data = np.load(index_path, allow_pickle=True)
        self.clip_ids = idx_data["clip_ids"]
        self.clip_idx = idx_data["clip_idx"]
        self.hand = idx_data["hand"]
        self.start = idx_data["start"]
        self.T_arr = idx_data["T"] if "T" in idx_data.files else None
        if mode == "eval":
            assert self.T_arr is not None, "eval 모드는 T 필드가 있는 dev/test index여야 합니다."

    @property
    def env(self):
        if self._env is None:
            self._env = _get_shared_lmdb_env(self.lmdb_path)
        return self._env

    def __len__(self):
        return len(self.clip_idx)

    def __getitem__(self, idx):
        clip_id = self.clip_ids[self.clip_idx[idx]]
        hand_code = int(self.hand[idx])
        side = _HAND_NAMES[hand_code]
        start = int(self.start[idx])
        context_len = (random.choice(self.context_len_choices)
                       if self.context_len_choices is not None else self.context_len)

        with self.env.begin(write=False) as txn:
            raw = txn.get(clip_id.encode("utf-8"))
        data = deserialize_minimal_npz(raw)

        feats = data[f"{side}_features"][:, :90]
        segment = data["segment"]
        if side == "left":
            feats = flip_left_hand_features(feats)

        gap_len = int(self.T_arr[idx]) if self.mode == "eval" else int(
            np.random.randint(self.gap_range[0], self.gap_range[1] + 1))
        window_len = context_len + gap_len + self.target_len

        T_clip = feats.shape[0]
        if start + window_len > T_clip:
            pad_len = start + window_len - T_clip
            feats = np.concatenate([feats, np.tile(feats[-1:], (pad_len, 1))], axis=0)
            segment = np.concatenate([segment, np.zeros(pad_len, dtype=segment.dtype)], axis=0)

        window = feats[start:start + window_len]
        segment_window = segment[start:start + window_len]

        known_mask = np.zeros(window_len, dtype=bool)
        known_mask[:context_len] = True
        known_mask[-self.target_len:] = True

        target_index = window_len - 1
        rel_pos = np.arange(window_len) - target_index

        item = {
            "features": torch.tensor(window, dtype=torch.float32),
            "known_mask": torch.tensor(known_mask, dtype=torch.bool),
            "rel_pos": torch.tensor(rel_pos, dtype=torch.long),
            "gap_len": gap_len,
            "clip_id": clip_id,
        }

        if self.keyframe_strategy == "temporal_midpoint":
            kp = select_keyframe_position_temporal_midpoint(context_len, gap_len)
            item["keyframe_pos"] = torch.tensor(kp, dtype=torch.long)
        elif self.keyframe_strategy == "segment2_priority":
            kp = select_keyframe_position_segment2_priority(context_len, gap_len, segment_window)
            item["keyframe_pos"] = torch.tensor(kp, dtype=torch.long)

        return item

In [ ]:
def collate_fn(batch):
    features = [b["features"] for b in batch]
    known_masks = [b["known_mask"] for b in batch]
    rel_pos_list = [b["rel_pos"] for b in batch]
    lengths = torch.tensor([f.shape[0] for f in features])

    features_padded = pad_sequence(features, batch_first=True)
    known_padded = pad_sequence(known_masks, batch_first=True, padding_value=False)
    rel_pos_padded = pad_sequence(rel_pos_list, batch_first=True, padding_value=0)

    T_max = features_padded.shape[1]
    valid_mask = torch.arange(T_max).unsqueeze(0) < lengths.unsqueeze(1)

    out = {
        "features": features_padded,
        "known_mask": known_padded,
        "rel_pos": rel_pos_padded,
        "valid_mask": valid_mask,
        "clip_ids": [b["clip_id"] for b in batch],
    }
    if "keyframe_pos" in batch[0]:
        out["keyframe_pos"] = torch.stack([b["keyframe_pos"] for b in batch])
    return out

In [ ]:
H2S_TRAIN_LMDB = f"{LOCAL_DATA_ROOT}/lmdb/train/How2Sign_reopt_train.lmdb"

train_dataset = IndexedHandInbetweeningDataset(
    H2S_TRAIN_LMDB, TRAIN_INDEX_PATH, mode="train", context_len=10, gap_range=(5, 30), target_len=1)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=0)  # A100 기준 (T4면 32로 낮추세요)

## 6. 입력 특징 변환 (build_silk_features)

회전값과 velocity를 concat해 180차원 입력으로 만듭니다 (신영님 SILK 파이프라인과 동일).

In [ ]:
def build_silk_features(features, known_mask):
    """transition(생성 대상) 구간을 0으로 채운 뒤 (회전값, velocity)를 concat."""
    mask3 = known_mask.unsqueeze(-1).float()
    pose_feat = features * mask3
    velocity = torch.zeros_like(pose_feat)
    velocity[:, 1:] = pose_feat[:, 1:] - pose_feat[:, :-1]
    velocity = velocity * mask3
    return torch.cat([pose_feat, velocity], dim=-1)

## 7. 모델 정의

SILK 구조 기반 Transformer. `d_model=1024, nhead=8, num_layers=6, dim_feedforward=4096`
(신영님의 `SILKSignSpark`, best_val_score 0.0493 체크포인트와 동일 설정).

In [ ]:
import torch.nn as nn


class RelativePositionalEncoding(nn.Module):
    """target(마지막 프레임) 기준 상대위치(rel_pos = 인덱스 - target_인덱스, 항상 <=0)를 입력에 더하는 임베딩."""

    def __init__(self, d_model, max_len=41):
        super().__init__()
        self.max_len = max_len
        self.pos_embedding = nn.Embedding(2 * max_len + 1, d_model)

    def forward(self, x, rel_pos):
        idx = (rel_pos + self.max_len).clamp(0, 2 * self.max_len)
        return x + self.pos_embedding(idx)


class SilkHandEncoderV2(nn.Module):
    def __init__(self, d_in=180, d_out=90, d_model=1024, nhead=8, num_layers=6,
                 dim_feedforward=4096, dropout=0.1, max_len=41):
        super().__init__()
        self.input_proj = nn.Linear(d_in, d_model)
        self.rel_pos_enc = RelativePositionalEncoding(d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_head = nn.Linear(d_model, d_out)

    def forward(self, x, rel_pos, valid_mask):
        h = self.input_proj(x)
        h = self.rel_pos_enc(h, rel_pos)
        key_padding_mask = ~valid_mask if valid_mask is not None else None
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        return self.output_head(h)

## 8. 손실 함수

In [ ]:
def silk_l1_loss(pred, target, known_mask, valid_mask):
    """생성 대상(transition)이면서 진짜 데이터인 프레임에서만 L1 loss 계산."""
    loss_mask = (~known_mask) & valid_mask
    diff = (pred - target).abs() * loss_mask.unsqueeze(-1)
    denom = loss_mask.sum() * pred.shape[-1]
    return diff.sum() / denom.clamp(min=1)

## 9. MANO 레이어 로드 + 평가지표 (L2Q · L2P-MANO · NPSS)

신영님의 MANO 설정(`flat_hand_mean=False`)과 동일하게 맞췄습니다.

In [ ]:
!pip install chumpy -q

import inspect
if not hasattr(np, "bool"): np.bool = bool
if not hasattr(np, "int"): np.int = int
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "object"): np.object = object
if not hasattr(np, "str"): np.str = str
if not hasattr(np, "complex"): np.complex = complex
if not hasattr(np, "unicode"): np.unicode = str
if not hasattr(inspect, "getargspec"): inspect.getargspec = inspect.getfullargspec
import chumpy
print("chumpy 로드 성공")

In [ ]:
import smplx

MANO_MODEL_ROOT = "/content/drive/MyDrive/KUBIG/contest"

mano_right = smplx.MANO(
    model_path=MANO_MODEL_ROOT, is_rhand=True,
    use_pca=False, flat_hand_mean=False, num_pca_comps=45,
).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
mano_right.eval()

DEVICE = next(mano_right.parameters()).device
N_JOINTS = 15
EVAL_T_VALUES = [5, 10, 20, 30]

In [ ]:
from scipy.spatial.transform import Rotation

def rotation_matrix_to_quaternion_np(mats):
    quats = Rotation.from_matrix(mats).as_quat()  # scipy: [x,y,z,w]
    return quats[:, [3, 0, 1, 2]]  # [w,x,y,z]로 재배열


def l2q_error(pred_90, gt_90, n_joints=N_JOINTS):
    T = pred_90.shape[0]
    pred_mats = _rot6d_to_matrix_np(pred_90.reshape(T * n_joints, 6))
    gt_mats = _rot6d_to_matrix_np(gt_90.reshape(T * n_joints, 6))
    pred_q = rotation_matrix_to_quaternion_np(pred_mats)
    gt_q = rotation_matrix_to_quaternion_np(gt_mats)
    dist = np.minimum(
        np.linalg.norm(pred_q - gt_q, axis=-1),
        np.linalg.norm(pred_q + gt_q, axis=-1),
    )  # double cover 보정
    return float(dist.mean())


def sixd_sequence_to_axis_angle(seq_6d):
    """(T, n_joints, 6) -> (T, n_joints, 3) axis-angle"""
    T, J, _ = seq_6d.shape
    mats = _rot6d_to_matrix_np(seq_6d.reshape(T * J, 6))
    aa = Rotation.from_matrix(mats).as_rotvec()
    return torch.tensor(aa.reshape(T, J, 3), dtype=torch.float32)


def mano_forward(mano_layer, hand_pose_aa, device=DEVICE):
    T = hand_pose_aa.shape[0]
    global_orient = torch.zeros(T, 3, device=device)
    hand_pose = hand_pose_aa.reshape(T, -1).to(device)
    betas = torch.zeros(T, 10, device=device)
    output = mano_layer(global_orient=global_orient, hand_pose=hand_pose, betas=betas, return_verts=True)
    return output.joints


def l2p_error_mano(pred_90, gt_90, mano_layer, n_joints=N_JOINTS):
    """실제 MANO forward kinematics 기반 위치 오차 (global_orient=0 고정, 손목 포함 평균)."""
    T = pred_90.shape[0]
    pred_aa = sixd_sequence_to_axis_angle(pred_90.reshape(T, n_joints, 6))
    gt_aa = sixd_sequence_to_axis_angle(gt_90.reshape(T, n_joints, 6))
    pred_joints = mano_forward(mano_layer, pred_aa).detach().cpu().numpy()
    gt_joints = mano_forward(mano_layer, gt_aa).detach().cpu().numpy()
    return float(np.linalg.norm(pred_joints - gt_joints, axis=-1).mean())


def npss(pred_seq, gt_seq):
    pred_fft = np.abs(np.fft.fft(pred_seq, axis=0)) ** 2
    gt_fft = np.abs(np.fft.fft(gt_seq, axis=0)) ** 2
    gt_norm = gt_fft / (gt_fft.sum(axis=0, keepdims=True) + 1e-8)
    pred_norm = pred_fft / (pred_fft.sum(axis=0, keepdims=True) + 1e-8)
    diff = np.abs(gt_norm - pred_norm).sum(axis=0)
    weight = gt_fft.sum(axis=0)
    weight = weight / (weight.sum() + 1e-8)
    return float((diff * weight).sum())

## 10. T-버킷 평가 루프 (dev / test)

In [ ]:
from tqdm import tqdm

def full_evaluate_single_pass(model, lmdb_path, index_path, T_values=EVAL_T_VALUES,
                               mano_layer=mano_right, batch_size=16, max_batches_per_T=None):
    model.eval()
    results = {}
    ds = IndexedHandInbetweeningDataset(lmdb_path, index_path, mode="eval")

    for T in T_values:
        T_indices = np.where(ds.T_arr == T)[0]
        sub = torch.utils.data.Subset(ds, T_indices)
        loader = DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

        l2q_list, l2p_list, npss_list = [], [], []
        with torch.no_grad():
            for bi, batch in enumerate(tqdm(loader, desc=f"평가 T={T}", leave=False)):
                if max_batches_per_T is not None and bi >= max_batches_per_T:
                    break
                features = batch["features"].to(DEVICE)
                known_mask = batch["known_mask"].to(DEVICE)
                valid_mask = batch["valid_mask"].to(DEVICE)
                rel_pos = batch["rel_pos"].to(DEVICE)

                x_feat = build_silk_features(features, known_mask)
                pred = model(x_feat, rel_pos, valid_mask)

                pred_np = pred.cpu().numpy()
                target_np = features.cpu().numpy()
                known_np = known_mask.cpu().numpy()
                valid_np = valid_mask.cpu().numpy()

                for b in range(pred_np.shape[0]):
                    idx = np.where((~known_np[b]) & valid_np[b])[0]
                    if len(idx) == 0:
                        continue
                    gt_seg, pred_seg = target_np[b, idx], pred_np[b, idx]
                    l2q_list.append(l2q_error(pred_seg, gt_seg))
                    l2p_list.append(l2p_error_mano(pred_seg, gt_seg, mano_layer))
                    npss_list.append(npss(pred_seg, gt_seg))

        results[T] = {"L2Q": np.mean(l2q_list), "L2P": np.mean(l2p_list),
                      "NPSS": np.mean(npss_list), "n": len(l2q_list)}
        print(f"T={T}: L2Q={results[T]['L2Q']:.4f} L2P={results[T]['L2P']:.4f} "
              f"NPSS={results[T]['NPSS']:.4f} (n={results[T]['n']})")

    return results

## 11. Baseline (Context Transformer) 학습

In [ ]:
import os

H2S_DEV_LMDB = f"{LOCAL_DATA_ROOT}/lmdb/dev/How2Sign_reopt_dev.lmdb"
H2S_TEST_LMDB = f"{LOCAL_DATA_ROOT}/lmdb/test/How2Sign_reopt_test.lmdb"

NUM_EPOCHS = 2
EVAL_EVERY_N_STEPS = 500
CKPT_DIR = f"{DRIVE_CONTEST_ROOT}/checkpoints"  # Drive에 저장 (로컬 디스크는 런타임 끊기면 사라짐)
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = f"{CKPT_DIR}/stage1_context_ckpt.pt"  # Section 12(Two-stage) Detail Transformer가 이 체크포인트를 로드

model = SilkHandEncoderV2().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

start_epoch = 0
step = 0
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    step = ckpt.get("step", 0)
    start_epoch = ckpt.get("epoch", 0)
    print(f"기존 체크포인트에서 이어서 시작: epoch={start_epoch}, step={step}")
else:
    print("체크포인트 없음 - 처음부터 시작")

loss_history = []
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    for batch in tqdm(train_loader, desc=f"epoch {epoch}"):
        features = batch["features"].to(DEVICE)
        known_mask = batch["known_mask"].to(DEVICE)
        valid_mask = batch["valid_mask"].to(DEVICE)
        rel_pos = batch["rel_pos"].to(DEVICE)

        x_feat = build_silk_features(features, known_mask)
        pred = model(x_feat, rel_pos, valid_mask)
        loss = silk_l1_loss(pred, features, known_mask, valid_mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        step += 1

        if step % EVAL_EVERY_N_STEPS == 0:
            full_evaluate_single_pass(model, H2S_DEV_LMDB, DEV_INDEX_PATH, max_batches_per_T=20)
            torch.save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                        "epoch": epoch, "step": step}, CKPT_PATH)
            model.train()

    print(f"[epoch {epoch}] 평균 loss: {np.mean(loss_history[-len(train_loader):]):.4f}")
    torch.save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                "epoch": epoch + 1, "step": step}, CKPT_PATH)

print("체크포인트 저장 완료:", CKPT_PATH)

In [ ]:
print("=== Baseline (Context Transformer 단독) 평가 (test) ===")
baseline_test_results = full_evaluate_single_pass(model, H2S_TEST_LMDB, TEST_INDEX_PATH,
                                                    batch_size=64, max_batches_per_T=240)

## 12. Two-stage Transformer (Qin et al., SIGGRAPH Asia 2022)

Context Transformer(=baseline, freeze) + Detail Transformer(신규 학습). Context의 거친(coarse) 예측으로
gap을 채운 뒤 Detail이 다듬습니다.

In [ ]:
# Detail 특징 변환 + Stage1 합성(composite)
def build_detail_features(composited_features, known_mask):
    velocity = torch.zeros_like(composited_features)
    velocity[:, 1:] = composited_features[:, 1:] - composited_features[:, :-1]
    return torch.cat([composited_features, velocity], dim=-1)

def composite_with_stage1(stage1_model, features, known_mask, rel_pos, valid_mask):
    with torch.no_grad():
        x1 = build_silk_features(features, known_mask)
        coarse_pred = stage1_model(x1, rel_pos, valid_mask)
    composited = torch.where(known_mask.unsqueeze(-1), features, coarse_pred)
    return composited

In [ ]:
# Context Transformer = Section 11의 baseline 체크포인트, 완전히 freeze
context_model = SilkHandEncoderV2().to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
context_model.load_state_dict(ckpt["model_state"])
context_model.eval()
for p in context_model.parameters():
    p.requires_grad_(False)
print("Context Transformer 로드 완료:", CKPT_PATH, "| epoch:", ckpt.get("epoch"), "| step:", ckpt.get("step"))

### Detail Transformer 학습 (Context는 고정)

In [ ]:
NUM_EPOCHS_DETAIL = 2
EVAL_EVERY_N_STEPS_DETAIL = 5000
CKPT_DIR = f"{DRIVE_CONTEST_ROOT}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
DETAIL_CKPT_PATH = f"{CKPT_DIR}/detail_transformer_ckpt.pt"

detail_model = SilkHandEncoderV2().to(DEVICE)
detail_optimizer = torch.optim.Adam(detail_model.parameters(), lr=1e-4)

start_epoch = 0
step = 0
if os.path.exists(DETAIL_CKPT_PATH):
    ckpt = torch.load(DETAIL_CKPT_PATH, map_location=DEVICE)
    detail_model.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        detail_optimizer.load_state_dict(ckpt["optimizer_state"])
    step = ckpt.get("step", 0)
    start_epoch = ckpt.get("epoch", 0)
    print(f"기존 체크포인트에서 이어서 시작: epoch={start_epoch}, step={step}")
else:
    print("체크포인트 없음 - 처음부터 시작")

for epoch in range(start_epoch, NUM_EPOCHS_DETAIL):
    detail_model.train()
    losses = []
    for batch in tqdm(train_loader, desc=f"detail epoch {epoch}"):
        features = batch["features"].to(DEVICE)
        known_mask = batch["known_mask"].to(DEVICE)
        valid_mask = batch["valid_mask"].to(DEVICE)
        rel_pos = batch["rel_pos"].to(DEVICE)

        composited = composite_with_stage1(context_model, features, known_mask, rel_pos, valid_mask)
        x_detail = build_detail_features(composited, known_mask)
        pred = detail_model(x_detail, rel_pos, valid_mask)
        loss = silk_l1_loss(pred, features, known_mask, valid_mask)

        detail_optimizer.zero_grad()
        loss.backward()
        detail_optimizer.step()
        losses.append(loss.item())
        step += 1

        if step % EVAL_EVERY_N_STEPS_DETAIL == 0:
            torch.save({"model_state": detail_model.state_dict(), "optimizer_state": detail_optimizer.state_dict(),
                        "epoch": epoch, "step": step}, DETAIL_CKPT_PATH)
    print(f"detail epoch {epoch} 평균 loss: {np.mean(losses):.4f}")
    torch.save({"model_state": detail_model.state_dict(), "optimizer_state": detail_optimizer.state_dict(),
                "epoch": epoch + 1, "step": step}, DETAIL_CKPT_PATH)

print("Detail Transformer 학습 완료:", DETAIL_CKPT_PATH)

### 2단계(Context→Detail) 추론 및 평가

In [ ]:
def two_stage_context_detail_predict(context_model, detail_model, features, known_mask, valid_mask, rel_pos):
    composited = composite_with_stage1(context_model, features, known_mask, rel_pos, valid_mask)
    x_detail = build_detail_features(composited, known_mask)
    with torch.no_grad():
        detail_pred = detail_model(x_detail, rel_pos, valid_mask)
    final_pred = torch.where(known_mask.unsqueeze(-1), features, detail_pred)
    return final_pred


def full_evaluate_context_detail(context_model, detail_model, lmdb_path, index_path, T_values=EVAL_T_VALUES,
                                  mano_layer=mano_right, batch_size=16, max_batches_per_T=None):
    context_model.eval()
    detail_model.eval()
    results = {}
    ds = IndexedHandInbetweeningDataset(lmdb_path, index_path, mode="eval")

    for T in T_values:
        T_indices = np.where(ds.T_arr == T)[0]
        sub = torch.utils.data.Subset(ds, T_indices)
        loader = DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

        l2q_list, l2p_list, npss_list = [], [], []
        with torch.no_grad():
            for bi, batch in enumerate(tqdm(loader, desc=f"Context+Detail 평가 T={T}", leave=False)):
                if max_batches_per_T is not None and bi >= max_batches_per_T:
                    break
                features = batch["features"].to(DEVICE)
                known_mask = batch["known_mask"].to(DEVICE)
                valid_mask = batch["valid_mask"].to(DEVICE)
                rel_pos = batch["rel_pos"].to(DEVICE)

                pred = two_stage_context_detail_predict(context_model, detail_model, features, known_mask,
                                                          valid_mask, rel_pos)

                pred_np = pred.cpu().numpy()
                target_np = features.cpu().numpy()
                known_np = known_mask.cpu().numpy()
                valid_np = valid_mask.cpu().numpy()

                for b in range(pred_np.shape[0]):
                    idx = np.where((~known_np[b]) & valid_np[b])[0]
                    if len(idx) == 0:
                        continue
                    gt_seg, pred_seg = target_np[b, idx], pred_np[b, idx]
                    l2q_list.append(l2q_error(pred_seg, gt_seg))
                    l2p_list.append(l2p_error_mano(pred_seg, gt_seg, mano_layer))
                    npss_list.append(npss(pred_seg, gt_seg))

        results[T] = {"L2Q": np.mean(l2q_list), "L2P": np.mean(l2p_list),
                      "NPSS": np.mean(npss_list), "n": len(l2q_list)}
        print(f"T={T}: L2Q={results[T]['L2Q']:.4f} L2P={results[T]['L2P']:.4f} "
              f"NPSS={results[T]['NPSS']:.4f} (n={results[T]['n']})")

    return results

In [ ]:
print("=== Context+Detail Two-stage Transformer 평가 (test) ===")
results_context_detail_test = full_evaluate_context_detail(context_model, detail_model,
                                                             H2S_TEST_LMDB, TEST_INDEX_PATH,
                                                             batch_size=64, max_batches_per_T=240)

## 13. 다중 키프레임 구조 결합

Two-stage 구조를 확장해, gap 안의 중간 지점(`segment==2` 라벨, 없으면 시간적 중점)을 추가 known 지점으로
노출시켜 하나의 gap을 두 개로 분할합니다. 이 위치의 값은 학습·추론 모두 Context Transformer의 예측값을
사용하고(학습·추론 조건 일치, exposure bias 방지), loss·평가에서는 이 위치를 제외합니다.

In [ ]:
# 키프레임 위치 선택
def select_keyframe_position_temporal_midpoint(context_len, gap_len):
    return context_len + gap_len // 2

def select_keyframe_position_segment2_priority(context_len, gap_len, segment_window):
    gap_slice = segment_window[context_len:context_len + gap_len]
    candidates = np.where(gap_slice == 2)[0]
    if len(candidates) == 0:
        return select_keyframe_position_temporal_midpoint(context_len, gap_len)
    mid = gap_len // 2
    best = candidates[np.argmin(np.abs(candidates - mid))]
    return context_len + int(best)

In [ ]:
# known_mask 확장: keyframe_pos 위치도 True로 추가 (loss/채점에서 제외할 범위)
def build_known_mask_multi(known_mask, keyframe_pos, valid_mask):
    known_mask_multi = known_mask.clone()
    B = known_mask.shape[0]
    batch_idx = torch.arange(B, device=known_mask.device)
    safe_pos = keyframe_pos.clamp(0, known_mask.shape[1] - 1)
    known_mask_multi[batch_idx, safe_pos] = True
    return known_mask_multi

In [ ]:
train_dataset_multi = IndexedHandInbetweeningDataset(
    H2S_TRAIN_LMDB, TRAIN_INDEX_PATH, mode="train", context_len=10, gap_range=(5, 30), target_len=1,
    keyframe_strategy="segment2_priority")
train_loader_multi = DataLoader(train_dataset_multi, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=0)

### Detail(다중 키프레임) 학습 (Context는 그대로 고정 재사용)

In [ ]:
NUM_EPOCHS_DETAIL_MULTI = 1
EVAL_EVERY_N_STEPS_MULTI = 5000
CKPT_DIR = f"{DRIVE_CONTEST_ROOT}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
DETAIL_MULTI_CKPT_PATH = f"{CKPT_DIR}/detail_multi_transformer_ckpt.pt"

detail_model_multi = SilkHandEncoderV2().to(DEVICE)
detail_multi_optimizer = torch.optim.Adam(detail_model_multi.parameters(), lr=1e-4)

start_epoch = 0
step = 0
if os.path.exists(DETAIL_MULTI_CKPT_PATH):
    ckpt = torch.load(DETAIL_MULTI_CKPT_PATH, map_location=DEVICE)
    detail_model_multi.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        detail_multi_optimizer.load_state_dict(ckpt["optimizer_state"])
    step = ckpt.get("step", 0)
    start_epoch = ckpt.get("epoch", 0)
    print(f"기존 체크포인트에서 이어서 시작: epoch={start_epoch}, step={step}")
else:
    print("체크포인트 없음 - 처음부터 시작")

for epoch in range(start_epoch, NUM_EPOCHS_DETAIL_MULTI):
    detail_model_multi.train()
    losses = []
    for batch in tqdm(train_loader_multi, desc=f"detail-multi epoch {epoch}"):
        features = batch["features"].to(DEVICE)
        known_mask = batch["known_mask"].to(DEVICE)
        valid_mask = batch["valid_mask"].to(DEVICE)
        rel_pos = batch["rel_pos"].to(DEVICE)
        keyframe_pos = batch["keyframe_pos"].to(DEVICE)

        known_mask_multi = build_known_mask_multi(known_mask, keyframe_pos, valid_mask)

        # 입력 합성은 Section 12와 동일 (원래 known_mask 기준)
        composited = composite_with_stage1(context_model, features, known_mask, rel_pos, valid_mask)
        x_detail = build_detail_features(composited, known_mask)
        pred = detail_model_multi(x_detail, rel_pos, valid_mask)
        # loss만 known_mask_multi 기준 (중간 키프레임 위치는 loss에서 제외)
        loss = silk_l1_loss(pred, features, known_mask_multi, valid_mask)

        detail_multi_optimizer.zero_grad()
        loss.backward()
        detail_multi_optimizer.step()
        losses.append(loss.item())
        step += 1

        if step % EVAL_EVERY_N_STEPS_MULTI == 0:
            torch.save({"model_state": detail_model_multi.state_dict(),
                        "optimizer_state": detail_multi_optimizer.state_dict(),
                        "epoch": epoch, "step": step}, DETAIL_MULTI_CKPT_PATH)
    print(f"detail-multi epoch {epoch} 평균 loss: {np.mean(losses):.4f}")
    torch.save({"model_state": detail_model_multi.state_dict(),
                "optimizer_state": detail_multi_optimizer.state_dict(),
                "epoch": epoch + 1, "step": step}, DETAIL_MULTI_CKPT_PATH)

print("Detail Transformer(다중 키프레임) 학습 완료:", DETAIL_MULTI_CKPT_PATH)

In [ ]:
def full_evaluate_context_detail_multi(context_model, detail_model, lmdb_path, index_path, T_values=EVAL_T_VALUES,
                                        mano_layer=mano_right, batch_size=16, max_batches_per_T=None):
    context_model.eval()
    detail_model.eval()
    results = {}
    ds = IndexedHandInbetweeningDataset(lmdb_path, index_path, mode="eval", keyframe_strategy="temporal_midpoint")

    for T in T_values:
        T_indices = np.where(ds.T_arr == T)[0]
        sub = torch.utils.data.Subset(ds, T_indices)
        loader = DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

        l2q_list, l2p_list, npss_list = [], [], []
        with torch.no_grad():
            for bi, batch in enumerate(tqdm(loader, desc=f"다중 키프레임 평가 T={T}", leave=False)):
                if max_batches_per_T is not None and bi >= max_batches_per_T:
                    break
                features = batch["features"].to(DEVICE)
                known_mask = batch["known_mask"].to(DEVICE)
                valid_mask = batch["valid_mask"].to(DEVICE)
                rel_pos = batch["rel_pos"].to(DEVICE)
                keyframe_pos = batch["keyframe_pos"].to(DEVICE)

                known_mask_multi = build_known_mask_multi(known_mask, keyframe_pos, valid_mask)

                pred = two_stage_context_detail_predict(context_model, detail_model, features, known_mask,
                                                          valid_mask, rel_pos)

                pred_np = pred.cpu().numpy()
                target_np = features.cpu().numpy()
                known_multi_np = known_mask_multi.cpu().numpy()
                valid_np = valid_mask.cpu().numpy()

                for b in range(pred_np.shape[0]):
                    idx = np.where((~known_multi_np[b]) & valid_np[b])[0]  # 중간 키프레임 위치는 채점에서 제외
                    if len(idx) == 0:
                        continue
                    gt_seg, pred_seg = target_np[b, idx], pred_np[b, idx]
                    l2q_list.append(l2q_error(pred_seg, gt_seg))
                    l2p_list.append(l2p_error_mano(pred_seg, gt_seg, mano_layer))
                    npss_list.append(npss(pred_seg, gt_seg))

        results[T] = {"L2Q": np.mean(l2q_list), "L2P": np.mean(l2p_list),
                      "NPSS": np.mean(npss_list), "n": len(l2q_list)}
        print(f"T={T}: L2Q={results[T]['L2Q']:.4f} L2P={results[T]['L2P']:.4f} "
              f"NPSS={results[T]['NPSS']:.4f} (n={results[T]['n']})")

    return results

In [ ]:
print("=== 다중 키프레임 Two-stage 평가 (test) ===")
results_context_detail_multi_test = full_evaluate_context_detail_multi(
    context_model, detail_model_multi, H2S_TEST_LMDB, TEST_INDEX_PATH, batch_size=64, max_batches_per_T=240)

## 14. MANO 시각화 (GT vs Baseline vs Two-stage vs 다중 키프레임)

T=5/10/20/30별로 test셋 클립을 하나씩 뽑아 MANO 관절 스켈레톤 GIF로 4개 모델을 비교합니다.

In [ ]:
# 한글 폰트 설정 (Colab 기본 matplotlib은 한글 미지원)
!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 다중 키프레임 최종 예측 함수
def multi_keyframe_predict(context_model, detail_model, features, known_mask, keyframe_pos, valid_mask, rel_pos):
    with torch.no_grad():
        x1 = build_silk_features(features, known_mask)
        coarse_pred = context_model(x1, rel_pos, valid_mask)
    composited = torch.where(known_mask.unsqueeze(-1), features, coarse_pred)
    x_detail = build_detail_features(composited, known_mask)
    with torch.no_grad():
        detail_pred = detail_model(x_detail, rel_pos, valid_mask)

    known_mask_multi = build_known_mask_multi(known_mask, keyframe_pos, valid_mask)
    keyframe_only_mask = known_mask_multi & (~known_mask)

    # 중간 키프레임 자리는 Context의 예측값, 나머지 gap은 Detail의 예측값, 원래 known 구간은 GT
    final_pred = torch.where(keyframe_only_mask.unsqueeze(-1), coarse_pred, detail_pred)
    final_pred = torch.where(known_mask.unsqueeze(-1), features, final_pred)
    return final_pred

In [ ]:
# 6D 예측값 -> MANO 관절 좌표 변환
MANO_PARENTS = [-1, 0,1,2, 0,4,5, 0,7,8, 0,10,11, 0,13,14]  # wrist + 손가락 5개x3관절 (index,middle,pinky,ring,thumb)

def get_mano_joints_np(pose_90_np):
    T = pose_90_np.shape[0]
    seq_6d = pose_90_np.reshape(T, N_JOINTS, 6)
    aa = sixd_sequence_to_axis_angle(seq_6d)
    joints = mano_forward(mano_right, aa).detach().cpu().numpy()
    return joints[:, :15, :]

In [ ]:
# 클립 하나 뽑아서 GT+3모델 4분할 GIF 만들기
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def get_sample_for_T(ds, T, sample_idx=0):
    T_indices = np.where(ds.T_arr == T)[0]
    item = ds[T_indices[sample_idx]]
    return collate_fn([item])

def make_comparison_gif(T, save_path, sample_idx=0):
    ds_eval = IndexedHandInbetweeningDataset(H2S_TEST_LMDB, TEST_INDEX_PATH, mode="eval",
                                              keyframe_strategy="temporal_midpoint")
    batch = get_sample_for_T(ds_eval, T, sample_idx=sample_idx)
    features = batch["features"].to(DEVICE)
    known_mask = batch["known_mask"].to(DEVICE)
    valid_mask = batch["valid_mask"].to(DEVICE)
    rel_pos = batch["rel_pos"].to(DEVICE)
    keyframe_pos = batch["keyframe_pos"].to(DEVICE)

    with torch.no_grad():
        x1 = build_silk_features(features, known_mask)
        baseline_pred = context_model(x1, rel_pos, valid_mask)
    baseline_final = torch.where(known_mask.unsqueeze(-1), features, baseline_pred)

    twostage_final = two_stage_context_detail_predict(context_model, detail_model, features, known_mask,
                                                        valid_mask, rel_pos)
    multi_final = multi_keyframe_predict(context_model, detail_model_multi, features, known_mask,
                                          keyframe_pos, valid_mask, rel_pos)

    sources = {
        "GT": features[0].cpu().numpy(),
        "Baseline": baseline_final[0].cpu().numpy(),
        "Two-stage": twostage_final[0].cpu().numpy(),
        "다중 키프레임": multi_final[0].cpu().numpy(),
    }
    joints_dict = {name: get_mano_joints_np(seq) for name, seq in sources.items()}

    T_len = sources["GT"].shape[0]
    all_pts = np.concatenate(list(joints_dict.values()), axis=0).reshape(-1, 3)
    center = all_pts.mean(axis=0)
    radius = np.abs(all_pts - center).max() * 1.2

    fig = plt.figure(figsize=(10, 10))
    names = list(joints_dict.keys())
    lines = {}
    for i, name in enumerate(names):
        ax = fig.add_subplot(2, 2, i + 1, projection="3d")
        ax.set_title(name)
        ax.set_xlim(center[0]-radius, center[0]+radius)
        ax.set_ylim(center[1]-radius, center[1]+radius)
        ax.set_zlim(center[2]-radius, center[2]+radius)
        lines[name] = [ax.plot([0], [0], [0], "o-", color="tab:blue")[0] for _ in MANO_PARENTS]

    def update(frame_idx):
        for name in names:
            pts = joints_dict[name][frame_idx]
            for j, p in enumerate(MANO_PARENTS):
                if p == -1:
                    lines[name][j].set_data([pts[0, 0]], [pts[0, 1]])
                    lines[name][j].set_3d_properties([pts[0, 2]])
                else:
                    lines[name][j].set_data([pts[p, 0], pts[j, 0]], [pts[p, 1], pts[j, 1]])
                    lines[name][j].set_3d_properties([pts[p, 2], pts[j, 2]])
        fig.suptitle(f"T={T} | frame {frame_idx+1}/{T_len}")
        return sum(lines.values(), [])

    anim = FuncAnimation(fig, update, frames=T_len, interval=150)
    anim.save(save_path, writer=PillowWriter(fps=6))
    plt.close(fig)
    print(f"저장 완료: {save_path}")

In [ ]:
for T in [5, 10, 20, 30]:
    make_comparison_gif(T, f"{DRIVE_CONTEST_ROOT}/viz_T{T}.gif")

## 15. 다중 키프레임 개선 시도 — loss 가중치 조정 (soft-loss)

중간 키프레임 위치를 loss에서 완전히 제외하는 대신 낮은 가중치(0.3)를 부여해봤습니다.
결과: Two-stage보다 모든 T·모든 지표에서 더 나빠 이 방향은 채택하지 않았습니다.

In [ ]:
def silk_weighted_l1_loss(pred, target, weight_mask, valid_mask):
    w = weight_mask * valid_mask.float()
    diff = (pred - target).abs() * w.unsqueeze(-1)
    denom = w.sum() * pred.shape[-1]
    return diff.sum() / denom.clamp(min=1e-8)

def build_loss_weight_multi(known_mask, keyframe_pos, keyframe_weight=0.3):
    B, Tm = known_mask.shape
    weight = (~known_mask).float()  # gap 전체 = 1.0, context/target = 0.0
    batch_idx = torch.arange(B, device=known_mask.device)
    safe_pos = keyframe_pos.clamp(0, Tm - 1)
    weight[batch_idx, safe_pos] = keyframe_weight
    return weight

In [ ]:
KEYFRAME_LOSS_WEIGHT = 0.3

NUM_EPOCHS_DETAIL_MULTI_SOFT = 1
EVAL_EVERY_N_STEPS_SOFT = 5000
CKPT_DIR = f"{DRIVE_CONTEST_ROOT}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
DETAIL_MULTI_SOFT_CKPT_PATH = f"{CKPT_DIR}/detail_multi_soft_transformer_ckpt.pt"

detail_model_multi_soft = SilkHandEncoderV2().to(DEVICE)
detail_multi_soft_optimizer = torch.optim.Adam(detail_model_multi_soft.parameters(), lr=1e-4)

start_epoch = 0
step = 0
if os.path.exists(DETAIL_MULTI_SOFT_CKPT_PATH):
    ckpt = torch.load(DETAIL_MULTI_SOFT_CKPT_PATH, map_location=DEVICE)
    detail_model_multi_soft.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        detail_multi_soft_optimizer.load_state_dict(ckpt["optimizer_state"])
    step = ckpt.get("step", 0)
    start_epoch = ckpt.get("epoch", 0)
    print(f"기존 체크포인트에서 이어서 시작: epoch={start_epoch}, step={step}")
else:
    print("체크포인트 없음 - 처음부터 시작")

for epoch in range(start_epoch, NUM_EPOCHS_DETAIL_MULTI_SOFT):
    detail_model_multi_soft.train()
    losses = []
    for batch in tqdm(train_loader_multi, desc=f"detail-multi-soft epoch {epoch}"):
        features = batch["features"].to(DEVICE)
        known_mask = batch["known_mask"].to(DEVICE)
        valid_mask = batch["valid_mask"].to(DEVICE)
        rel_pos = batch["rel_pos"].to(DEVICE)
        keyframe_pos = batch["keyframe_pos"].to(DEVICE)

        composited = composite_with_stage1(context_model, features, known_mask, rel_pos, valid_mask)
        x_detail = build_detail_features(composited, known_mask)
        pred = detail_model_multi_soft(x_detail, rel_pos, valid_mask)

        loss_weight = build_loss_weight_multi(known_mask, keyframe_pos, keyframe_weight=KEYFRAME_LOSS_WEIGHT)
        loss = silk_weighted_l1_loss(pred, features, loss_weight, valid_mask)

        detail_multi_soft_optimizer.zero_grad()
        loss.backward()
        detail_multi_soft_optimizer.step()
        losses.append(loss.item())
        step += 1

        if step % EVAL_EVERY_N_STEPS_SOFT == 0:
            torch.save({"model_state": detail_model_multi_soft.state_dict(),
                        "optimizer_state": detail_multi_soft_optimizer.state_dict(),
                        "epoch": epoch, "step": step}, DETAIL_MULTI_SOFT_CKPT_PATH)
    print(f"detail-multi-soft epoch {epoch} 평균 loss: {np.mean(losses):.4f}")
    torch.save({"model_state": detail_model_multi_soft.state_dict(),
                "optimizer_state": detail_multi_soft_optimizer.state_dict(),
                "epoch": epoch + 1, "step": step}, DETAIL_MULTI_SOFT_CKPT_PATH)

print("Detail Transformer(다중 키프레임, soft-loss) 학습 완료:", DETAIL_MULTI_SOFT_CKPT_PATH)

In [ ]:
print("=== 다중 키프레임 (soft-loss, keyframe_weight=0.3) 평가 (test) ===")
results_multi_soft_test = full_evaluate_context_detail(
    context_model, detail_model_multi_soft, H2S_TEST_LMDB, TEST_INDEX_PATH, batch_size=64, max_batches_per_T=240)